# Production Document Q&A with Dewey's Managed RAG Backend

Building a RAG pipeline from scratch requires wiring together many moving parts: a document parser, a chunking strategy, an embedding model, a vector database, a keyword search index, and a retrieval layer that reconciles them. Each piece has configuration to tune, infrastructure to host, and failure modes to handle.

**[Dewey](https://meetdewey.com)** is a managed document backend that handles this entire pipeline behind a single REST API. You upload a document and Dewey takes care of conversion, section extraction, chunking, embedding, and hybrid retrieval — leaving you to focus on the application layer.

In this notebook you will:

1. Upload a set of AI research papers to a Dewey collection
2. Run hybrid semantic + keyword search over them
3. Use section-aware retrieval to navigate document structure
4. Stream a grounded, cited answer from Dewey's agentic research endpoint
5. Build a simple RAG chat loop using the OpenAI Python SDK

**Prerequisites**: A Dewey API key (free at [meetdewey.com](https://meetdewey.com)) and an OpenAI API key.

## Setup

In [ ]:
%pip install meetdewey openai requests --quiet

In [ ]:
import os
import time
import textwrap
from pathlib import Path

import requests
from dewey import DeweyClient
from openai import OpenAI

DEWEY_API_KEY = os.environ.get("DEWEY_API_KEY", "dwy_live_...")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")

dewey = DeweyClient(api_key=DEWEY_API_KEY)
oai = OpenAI(api_key=OPENAI_API_KEY)

## 1. Create a collection and upload documents

A **collection** is Dewey's top-level namespace — analogous to a table in a database or an index in a search engine. All documents, sections, and chunks live inside a collection, and retrieval is always scoped to one.

We'll upload three foundational AI papers from ArXiv as PDFs.

In [ ]:
# Create a collection
collection = dewey.collections.create("AI Research Papers")
print(f"Collection created: {collection.id}")

In [ ]:
# Papers to upload: (filename, ArXiv PDF URL)
PAPERS = [
    ("attention-is-all-you-need.pdf",   "https://arxiv.org/pdf/1706.03762"),
    ("gpt3-language-models.pdf",         "https://arxiv.org/pdf/2005.14165"),
    ("rag-knowledge-intensive-nlp.pdf",  "https://arxiv.org/pdf/2005.11401"),
]

doc_ids = []
for filename, url in PAPERS:
    print(f"Uploading {filename}...", end=" ")
    pdf_bytes = requests.get(url, timeout=30).content
    doc = dewey.documents.upload(
        collection.id,
        pdf_bytes,
        filename=filename,
        content_type="application/pdf",
    )
    doc_ids.append(doc.id)
    print(f"id={doc.id}")

print(f"\nUploaded {len(doc_ids)} documents.")

Dewey processes documents asynchronously. The pipeline runs conversion → section extraction → chunking → embedding in the background. We poll until all documents reach `ready` status.

In [ ]:
def wait_for_ready(collection_id: str, doc_ids: list[str], poll_interval: float = 5.0) -> None:
    """Block until all documents in doc_ids reach 'ready' or 'error' status."""
    pending = set(doc_ids)
    while pending:
        still_pending = set()
        for doc_id in pending:
            doc = dewey.documents.get(collection_id, doc_id)
            if doc.status == "error":
                raise RuntimeError(f"{doc.filename} failed: {doc.error_message}")
            if doc.status != "ready":
                still_pending.add(doc_id)
        if still_pending:
            print(f"  waiting — {len(still_pending)}/{len(doc_ids)} still processing...")
            time.sleep(poll_interval)
        pending = still_pending
    print("All documents ready.")

wait_for_ready(collection.id, doc_ids)

### Inspect the document structure

Dewey parses each document's heading hierarchy into **sections** — a first-class API primitive. Sections with generic titles ("Introduction", "Chapter 3") automatically receive AI-generated summaries so they remain findable.

In [ ]:
# List the top-level sections for the first document
sections = dewey.sections.list(collection.id, doc_ids[0])
print(f"Sections in '{PAPERS[0][0]}':")
for s in sections[:10]:
    indent = "  " * (s.level - 1)
    summary_preview = f" — {s.summary[:80]}..." if s.summary else ""
    print(f"  {indent}[L{s.level}] {s.title}{summary_preview}")

## 2. Hybrid search

Dewey's retrieval endpoint runs **hybrid search**: dense vector similarity (via text-embedding-3-small by default) combined with BM25 keyword search, merged using Reciprocal Rank Fusion (RRF). This catches both semantic matches and exact-term matches that pure vector search misses.

Each result carries citation metadata — filename, section title, and section level — so you always know where a chunk came from.

In [ ]:
query = "How does the attention mechanism scale with sequence length?"
results = dewey.retrieval.query(collection.id, query, limit=5)

print(f"Top results for: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"  [{i}] score={r.score:.4f}  {r.document.filename} › {r.section.title}")
    print(f"      {r.chunk.content[:200].strip()}...")
    print()

## 3. Section-aware retrieval

In addition to chunk-level retrieval, Dewey exposes a **section scan** endpoint that searches section titles and summaries — not chunk content. This is useful when you want to identify which parts of a document are relevant before fetching their full text.

The mental model: **scan sections cheaply → fetch chunks only where needed**. This is especially valuable for large documents where loading every chunk into an LLM context is expensive.

In [ ]:
# Scan section titles and summaries first
scan_results = dewey.sections.scan(
    collection.id,
    query="multi-head attention architecture",
    top_k=5,
)

print("Relevant sections (titles + summaries only — no chunk content loaded yet):\n")
for r in scan_results:
    print(f"  {r.filename} › {r.title}  ({r.chunk_count} chunks)")
    if r.summary:
        print(f"    {r.summary[:150]}...")
    print()

In [ ]:
# Fetch the full content of the most relevant section
top_section_id = scan_results[0].section_id
section = dewey.sections.get(top_section_id)

print(f"Full content of '{section.title}' ({len(section.content)} chars):\n")
print(textwrap.fill(section.content[:800], width=90))
print("...")

## 4. Agentic research with streaming

Dewey's `/research` endpoint runs a multi-step agentic loop: it searches, reads relevant sections, reasons across documents, and produces a grounded answer with full source citations. This is the equivalent of a research analyst skimming your document library — not just a single retrieval call.

The response streams over Server-Sent Events. Four depth levels trade cost for thoroughness:

| Depth | Credits | Use case |
|---|---|---|
| `quick` | 0.5 | Fast factual lookups |
| `balanced` | 1 | General Q&A (default) |
| `deep` | 3 | Multi-document synthesis |
| `exhaustive` | 8 | Comprehensive research |

`deep` and `exhaustive` require a configured provider key (see section 5).

In [ ]:
question = (
    "Compare how the Transformer architecture and RAG each address the limitations "
    "of earlier sequence-to-sequence models. What problems does each solve?"
)

print(f"Question: {question}\n")
print("=" * 80)

answer_parts = []
sources = []

for event in dewey.research.stream(collection.id, question, depth="balanced"):
    if event.type == "tool_call":
        print(f"  [searching] {event.query}", flush=True)
    elif event.type == "chunk":
        print(event.content, end="", flush=True)
        answer_parts.append(event.content)
    elif event.type == "done":
        sources = event.sources

print("\n" + "=" * 80)
print(f"\nSources ({len(sources)}):")
for s in sources:
    print(f"  {s.filename} › {s.section_title}")

## 5. Bring your own OpenAI key (BYOK)

By default, Dewey uses a managed OpenAI key and charges credits per research request. If you configure your own OpenAI key, Dewey routes all generation calls through it directly — no markup, and credit metering is bypassed entirely.

This makes sense once you want cost transparency and control, or when you want to use `deep` or `exhaustive` depth.

In [ ]:
# To register your OpenAI key with Dewey, you need your project ID.
# Find it in the Dewey dashboard or via the API:
#
#   provider_key = dewey.provider_keys.create(
#       project_id="<your-project-id>",
#       provider="openai",
#       key=OPENAI_API_KEY,
#       name="my-openai-key",
#   )
#
# Once registered, all research requests using this project's API key will route
# through your OpenAI account. You pay OpenAI directly at cost.

print("BYOK is not yet configured — generation will use Dewey's managed key and consume credits.")
print("To enable BYOK: fill in your project ID above, uncomment the provider_key.create(...)")
print("block, and run this cell. Once registered, deep/exhaustive depths are unlocked")
print("and credit metering is bypassed for this project.")

## 6. Building a RAG chat loop with the OpenAI SDK

You can also use Dewey purely as a retrieval backend and drive generation yourself with the OpenAI SDK. This gives you full control over prompting, model selection, and conversation management.

The pattern: **Dewey handles retrieval → OpenAI handles generation**.

In [ ]:
SYSTEM_PROMPT = """\
You are a research assistant with access to a library of AI papers.
Answer questions using only the provided context. Always cite your sources
by referencing the document filename and section title.
If the context does not contain enough information, say so."""


def rag_answer(question: str, k: int = 6) -> str:
    """Retrieve relevant chunks from Dewey and generate an answer with OpenAI."""
    # 1. Retrieve
    results = dewey.retrieval.query(collection.id, question, limit=k)

    # 2. Build context block
    context_parts = []
    for r in results:
        context_parts.append(
            f"[Source: {r.document.filename} › {r.section.title}]\n{r.chunk.content}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # 3. Generate
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
    )
    return response.choices[0].message.content


# Try it
answer = rag_answer("What is the role of positional encoding in the Transformer?")
print(textwrap.fill(answer, width=90))

In [ ]:
# Simple multi-turn chat loop
# Uncomment and run to interact in the notebook

# while True:
#     q = input("\nYou: ").strip()
#     if q.lower() in {"exit", "quit", "q"}:
#         break
#     answer = rag_answer(q)
#     print(f"\nAssistant: {answer}")

## 7. Cleanup

In [ ]:
# Delete the collection and all its documents when you're done
dewey.collections.delete(collection.id)
print(f"Collection {collection.id} deleted.")

## Summary

In this notebook you used Dewey to:

- **Ingest** PDF documents into a managed pipeline (convert → section → chunk → embed)
- **Search** with hybrid BM25 + vector retrieval that returns chunk-level results with full citation metadata
- **Navigate** document structure using section scan before committing to full chunk retrieval
- **Research** across multiple documents with a streaming agentic loop that produces grounded, cited answers
- **Integrate** Dewey's retrieval layer into an OpenAI-powered RAG chat loop

The full REST API and Python/TypeScript SDKs are documented at [meetdewey.com/docs](https://meetdewey.com/docs). A free tier is available with no credit card required.